In [ ]:
import sys
import importlib
# Add your local directory to the beginning of sys.path
sys.path.insert(0, '/Users/hannahzhou/Downloads/super_research/NNPIV')
sys.path.append('/Users/hannahzhou/Downloads/super_research/project_star_copy/src/')
sys.path.append('/Users/hannahzhou/Downloads/super_research/project_star_copy/src/models')
# sys.path.append('/Users/hannahzhou/Downloads/super_research/project_star_copy/src/datasets')
import data_loader
import LTMeanEmbedding
importlib.reload(data_loader)
importlib.reload(LTMeanEmbedding)
from LTMeanEmbedding import init_LTTrainDataSet
from data_loader import get_star_test_score_data, get_nyc_test_score_data
import numpy as np

grade_level = 3
outcome_type = "avsum"
lt_train = init_LTTrainDataSet(
        star_data = get_star_test_score_data(), 
        nyc_data = get_nyc_test_score_data(),
        grade_level = grade_level, 
        outcome_type = outcome_type)

In [102]:
import sys
import importlib

from sklearn.linear_model import LinearRegression
# Add your local directory to the beginning of sys.path
sys.path.insert(0, '/Users/hannahzhou/Downloads/super_research/NNPIV')
sys.path.append('/Users/hannahzhou/Downloads/super_research/project_star_copy/src/')
sys.path.append('/Users/hannahzhou/Downloads/super_research/project_star_copy/src/models')
# sys.path.append('/Users/hannahzhou/Downloads/super_research/project_star_copy/src/datasets')
import data_loader
import LTMeanEmbedding
importlib.reload(data_loader)
importlib.reload(LTMeanEmbedding)
from LTMeanEmbedding import init_LTTrainDataSet
from data_loader import get_star_test_score_data, get_nyc_test_score_data
import numpy as np
from nnpiv.tsls import tsls

outcome_type = "avsum"

for grade_level in range(3, 9):
    print(f"Grade Level: {grade_level}")
    lt_train = init_LTTrainDataSet(
        star_data = get_star_test_score_data(), 
        nyc_data = get_nyc_test_score_data(),
        grade_level = grade_level, 
        outcome_type = outcome_type)
    
    exp_outcome = np.full(lt_train.exp_covariate.shape[0], np.nan)[:, np.newaxis]
    Y = np.concatenate([exp_outcome, lt_train.obs_outcome])
    D = np.concatenate([lt_train.exp_treatment, lt_train.obs_treatment])
    S = np.concatenate([lt_train.exp_surrogate, lt_train.obs_surrogate])
    exp_group_indicator = np.full(lt_train.exp_covariate.shape[0], 0)[:, np.newaxis]
    obs_group_indicator = np.full(lt_train.obs_covariate.shape[0], 1)[:, np.newaxis]
    G = np.concatenate([exp_group_indicator, obs_group_indicator])
    X1 = np.concatenate([lt_train.exp_covariate, lt_train.obs_covariate])

    print(f"Num experimental samples: {lt_train.exp_covariate.shape[0]}")
    print(f"Num observational samples: {lt_train.obs_covariate.shape[0]}")

    from nnpiv.semiparametrics.dml_longterm_seq import DML_longterm_seq
    import importlib
    from nnpiv.semiparametrics import dml_longterm_seq
    importlib.reload(dml_longterm_seq)
    from nnpiv.rkhs import ApproxRKHSIVCV

    with open(f"grade_{grade_level}_discrete_results_tsls.txt", "w") as file:
        for d_discrete in range(12, 29):
            print(f"d_discrete: {d_discrete}")
            dml_longterm_rkhs = DML_longterm_seq(
                Y=Y,
                D=D,
                S=S,
                G=G,
                X1=X1,
                estimator="OR",
                longterm_model="surrogacy",
                verbose=True,
                # model1=ApproxRKHSIVCV(kernel_approx='nystrom', n_components=10,
                #                     kernel='rbf', gamma=.1, delta_scale='auto',
                #                     delta_exp=.4, alpha_scales=np.geomspace(1, 10000, 10), cv=5),
                model1=tsls(),
                # model2=ApproxRKHSIVCV(kernel_approx='nystrom', n_components=10,
                #                     kernel='rbf', gamma=.1, delta_scale='auto',
                #                     delta_exp=.4, alpha_scales=np.geomspace(1, 10000, 10), cv=5)
                model2=tsls(),
            )
            theta_hat, theta_var_hat, theta_cov_hat, confidence_interval = dml_longterm_rkhs.dml(d_discrete=d_discrete)
            # print(f"d_discrete: {d_discrete}, theta_hat: {theta_hat}, confidence_interval: {confidence_interval}")
            file.write(f"d_discrete: {d_discrete}, theta_hat: {theta_hat}, confidence_interval: {confidence_interval}\n")


Grade Level: 3


/var/folders/w2/1hs0b9n10qvc01qxl1y2n_q00000gn/T/ipykernel_11033/2217739641.py:50: DeprecationWarning: DML_longterm_seq is deprecated and will be removed in a future version. Use DML_longterm instead.
  dml_longterm_rkhs = DML_longterm_seq(


Num experimental samples: 2052
Num observational samples: 10223
d_discrete: 12
Rep: 1


100%|██████████| 5/5 [00:00<00:00, 95.18it/s]
/var/folders/w2/1hs0b9n10qvc01qxl1y2n_q00000gn/T/ipykernel_11033/2217739641.py:50: DeprecationWarning: DML_longterm_seq is deprecated and will be removed in a future version. Use DML_longterm instead.
  dml_longterm_rkhs = DML_longterm_seq(


psi_hat_array shape: (12275, 1)
theta_hat: [0.12502601], theta_var_hat: [0.00193057], theta_cov_hat: 0.0019305697997038696
Calculating confidence intervals with n=12275, alpha=0.05, ci_type=pointwise
d_discrete: 13
Rep: 1


...

psi_hat_array shape: (12275, 1)
theta_hat: [-0.38484694], theta_var_hat: [0.0169897], theta_cov_hat: 0.01698970362695002
Calculating confidence intervals with n=12275, alpha=0.05, ci_type=pointwise
d_discrete: 27
Rep: 1


100%|██████████| 5/5 [00:00<00:00, 141.41it/s]
/var/folders/w2/1hs0b9n10qvc01qxl1y2n_q00000gn/T/ipykernel_11033/2217739641.py:50: DeprecationWarning: DML_longterm_seq is deprecated and will be removed in a future version. Use DML_longterm instead.
  dml_longterm_rkhs = DML_longterm_seq(


psi_hat_array shape: (12275, 1)
theta_hat: [-0.53492401], theta_var_hat: [0.01324058], theta_cov_hat: 0.013240579996196556
Calculating confidence intervals with n=12275, alpha=0.05, ci_type=pointwise
d_discrete: 28
Rep: 1


 40%|████      | 2/5 [00:00<00:00,  8.05it/s]

ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by LinearRegression.

In [3]:
import numpy as np
exp_outcome = np.full(lt_train.exp_covariate.shape[0], np.nan)[:, np.newaxis]
Y = np.concatenate([exp_outcome, lt_train.obs_outcome])
D = np.concatenate([lt_train.exp_treatment, lt_train.obs_treatment])
S = np.concatenate([lt_train.exp_surrogate, lt_train.obs_surrogate])
exp_group_indicator = np.full(lt_train.exp_covariate.shape[0], 0)[:, np.newaxis]
obs_group_indicator = np.full(lt_train.obs_covariate.shape[0], 1)[:, np.newaxis]
G = np.concatenate([exp_group_indicator, obs_group_indicator])
X1 = np.concatenate([lt_train.exp_covariate, lt_train.obs_covariate])
print(Y.shape)
print(D.shape)
print(S.shape)
print(G.shape)
print(X1.shape)

(12275, 1)
(12275, 1)
(12275, 1)
(12275, 1)
(12275, 1)


In [6]:
print(dml_longterm_seq.__file__)

/Users/hannahzhou/Downloads/super_research/NNPIV/nnpiv/semiparametrics/dml_longterm_seq.py


In [46]:
from nnpiv.semiparametrics.dml_longterm_seq import DML_longterm_seq
import importlib
from nnpiv.semiparametrics import dml_longterm_seq
importlib.reload(dml_longterm_seq)
from nnpiv.rkhs import ApproxRKHSIVCV

dml_longterm_rkhs = DML_longterm_seq(
    Y=Y,
    D=D,
    S=S,
    G=G,
    X1=X1,
    estimator="OR",
    longterm_model="surrogacy",
    verbose=True,
    # n_components=10,
    model1=ApproxRKHSIVCV(kernel_approx='nystrom', n_components=10,
                           kernel='rbf', gamma=.1, delta_scale='auto',
                           delta_exp=.4, alpha_scales=np.geomspace(1, 10000, 10), cv=5),
    model2=ApproxRKHSIVCV(kernel_approx='nystrom', n_components=10,
                           kernel='rbf', gamma=.1, delta_scale='auto',
                           delta_exp=.4, alpha_scales=np.geomspace(1, 10000, 10), cv=5)
)

/var/folders/w2/1hs0b9n10qvc01qxl1y2n_q00000gn/T/ipykernel_11033/1865002338.py:7: DeprecationWarning: DML_longterm_seq is deprecated and will be removed in a future version. Use DML_longterm instead.
  dml_longterm_rkhs = DML_longterm_seq(


In [47]:
theta_hat, theta_var_hat, theta_cov_hat = dml_longterm_rkhs.dml(d_discrete=12)

Rep: 1


100%|██████████| 5/5 [00:00<00:00, 29.32it/s]

Fitting outcome model with surrogacy assumption...Fitting outcome model with surrogacy assumption...
Fitting first stage model...

Fitting first stage model...
Fitting outcome model with surrogacy assumption...
Fitting first stage model...
Shape of S: (9820, 1), X: (9820, 1), Y: (9820, 1), G: (9820, 1)
Fitting outcome model with surrogacy assumption...
Fitting first stage model...
Fitting outcome model with surrogacy assumption...
Fitting first stage model...
Shape of S1: (22, 1), X1: (22, 1), Y1: (22, 1)
Transforming data with polynomial features...
A1 shape: (22, 2)
Y1 shape: (22, 1)
Shape of S: (9820, 1), X: (9820, 1), Y: (9820, 1), G: (9820, 1)
Shape of S1: (22, 1), X1: (22, 1), Y1: (22, 1)
Transforming data with polynomial features...
A1 shape: (22, 2)
Y1 shape: (22, 1)
Shape of S: (9820, 1), X: (9820, 1), Y: (9820, 1), G: (9820, 1)
Shape of S1: (22, 1), X1: (22, 1), Y1: (22, 1)
Transforming data with polynomial features...
A1 shape: (22, 2)
Y1 shape: (22, 1)
Shape of S: (9820, 1)

In [50]:
print(theta_hat)
print(theta_var_hat)


[-0.12503456]
[0.00432197]


In [38]:
len(result[0][0])

2455

In [41]:
len(result[0][1])

2455

In [43]:
result[0][0].mean()

-0.13292428140021126

In [28]:
print(dml_longterm_rkhs.dml(d_discrete=12))

Rep: 1


 20%|██        | 1/5 [00:00<00:00,  5.66it/s]

Fitting outcome model with surrogacy assumption...Fitting outcome model with surrogacy assumption...
Fitting first stage model...

Fitting first stage model...
Fitting outcome model with surrogacy assumption...
Fitting first stage model...
Shape of S: (9820, 1), X: (9820, 1), Y: (9820, 1), G: (9820, 1)
Shape of S: (9820, 1), X: (9820, 1), Y: (9820, 1), G: (9820, 1)
Fitting outcome model with surrogacy assumption...
Fitting first stage model...
Fitting outcome model with surrogacy assumption...
Fitting first stage model...
Shape of S1: (23, 1), X1: (23, 1), Y1: (23, 1)
Transforming data with polynomial features...
A1 shape: (23, 2)
Y1 shape: (23, 1)
Shape of S1: (22, 1), X1: (22, 1), Y1: (22, 1)
Transforming data with polynomial features...
A1 shape: (22, 2)
Y1 shape: (22, 1)
Shape of S: (9820, 1), X: (9820, 1), Y: (9820, 1), G: (9820, 1)
Shape of S1: (22, 1), X1: (22, 1), Y1: (22, 1)
Transforming data with polynomial features...
A1 shape: (22, 2)
Y1 shape: (22, 1)
Shape of S: (9820, 1)

100%|██████████| 5/5 [00:00<00:00, 21.84it/s]



[[array([[-0.13292428],
       [-0.13292428],
       [-0.13292428],
       ...,
       [-0.13292428],
       [-0.13292428],
       [-0.13292428]]), array([[-0.20317514],
       [-0.20317514],
       [-0.20317514],
       ...,
       [-0.20317514],
       [-0.20317514],
       [-0.20317514]]), array([[-0.16393403],
       [-0.16393403],
       [-0.16393403],
       ...,
       [-0.16393403],
       [-0.16393403],
       [-0.16393403]]), array([[-0.11790047],
       [-0.11790047],
       [-0.11790047],
       ...,
       [-0.11790047],
       [-0.11790047],
       [-0.11790047]]), array([[-0.00723888],
       [-0.00723888],
       [-0.00723888],
       ...,
       [-0.00723888],
       [-0.00723888],
       [-0.00723888]])]]
